In [ ]:
#@title 按這裡開始（先按 ▶）
# ←投影片未含，執行所需
print('✅ W16 出發！本週目標：看穿正確率，學會用混淆矩陣與四個指標判斷模型好不好')
print('四個任務：改不平衡比例 → 印混淆矩陣 → 掃判斷門檻 → 故意做一次資料洩漏')
print('這一堂只吃 Colab，不用相機、不用感測器，全程只要改標了「← 改這裡」的那一行')

# W16 模型評估與常見陷阱（手機版）

手機開這本請先切成「電腦版網站」（iPhone：網址列 ᴀA → 要求電腦版網站；Android：右上 ⋮ → 電腦版網站），再點每一格左邊的 ▶ 執行。

本週要做四個實驗，讓正確率自己現出原形：

| 任務 | 你要做的事 | 要交什麼 |
|---|---|---|
| 一 | 改不平衡比例，看正確率怎麼騙人 | 截圖 |
| 二 | 把混淆矩陣四格印出來 | 截圖 |
| 三 | 掃判斷門檻，看兩個指標此消彼長 | 截圖＋門檻紀錄表 |
| 四 | 故意做一次資料洩漏再修好 | 截圖 |

**每次只換一個變因**，才看得出是誰影響了分數——這是做實驗的基本紀律。

## 任務一：改不平衡比例，看正確率怎麼騙人

**怎麼做**：只改 `bad` 那一行，其他都不要動，按 ▶ 就好。

1. 先跑預設值 `bad = 1`，記下**正確率**與**全猜好品基準**兩個數字。
2. 改成 `5` 再跑一次，看兩個數字各變多少。
3. 改成 `20` 再跑一次，這一次差距會拉開。
4. 寫一句結論：什麼情況下的正確率最不能相信。

**兩個數字越接近，代表這個模型越沒有做事。**

In [ ]:
#@title 第 1 格：改不平衡比例
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
bad = 1  #@param {type:"integer"}  # ← 改這裡：壞品佔百分之幾
X, y = make_classification(n_samples=2000, n_informative=5,
                           weights=[1 - bad/100], random_state=0)
m = LogisticRegression(max_iter=1000).fit(X[:1500], y[:1500])
print('模型正確率 =', round(m.score(X[1500:], y[1500:]), 3))
print('全猜好品基準 =', round(1 - y[1500:].mean(), 3))

## 任務二：把混淆矩陣四格印出來

**怎麼做**：這一格不用改任何東西，直接按 ▶，對照理論那張表看。

記法：**第二個字母是模型說什麼**（P 說是、N 說不是），**第一個字母是說得對不對**（T 對、F 錯）。
所以 FP 就是「模型說是，但說錯了」。

**先做這個檢查**：四格加起來要等於 500（測試資料筆數），不等於就是哪裡看錯了。

In [ ]:
#@title 第 2 格：把混淆矩陣印出來
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
p = m.predict(X[1500:])
tn, fp, fn, tp = confusion_matrix(y[1500:], p).ravel()
print('TP 抓對', tp, ' FP 誤報', fp)
print('FN 漏掉', fn, ' TN 判對', tn)
print(classification_report(y[1500:], p, digits=3))

## 任務三：掃判斷門檻，看兩個指標怎麼此消彼長

**怎麼做**：只改 `th` 那一行，四個門檻各跑一次，把數字填進紀錄表。

⚠️ **做這一段之前，先把第 1 格的 `bad` 改成 5 或 20 再重跑一次**。`bad = 1` 時壞品只有十來筆，四個門檻的數字幾乎不會動，看不出此消彼長。

1. 先跑預設 `0.5`，記下精確率與召回率，寫進紀錄表第二列。
2. 改成 `0.3`：抓得比較多，看哪一個指標上升了。
3. 改成 `0.7`：抓得比較少，看哪一個指標下降了。
4. 改成 `0.9`：會發生什麼極端狀況？也要記下來。
5. 挑一個門檻並說理由：如果是疾病篩檢，你會選哪一個？

**門檻紀錄表（四列都填滿才看得出趨勢，當堂收）**

| 判斷門檻 | 精確率 | 召回率 | 你看到什麼 |
|---|---|---|---|
| 0.3 | 　 | 　 | 門檻低：抓得多，誤報也跟著多 |
| 0.5 | 　 | 　 | 預設值，先記下來當基準 |
| 0.7 | 　 | 　 | 門檻高：說是的比較準，但開始漏抓 |
| 0.9 | 　 | 　 | 可能幾乎都不抓，召回率接近 0 |
| 趨勢 | 往上 | 往下 | 門檻調高時，兩個指標的走向 |
| 你的結論 | 　 | 　 | 如果是疾病篩檢，你會選哪一個門檻 |

In [ ]:
#@title 第 3 格：掃判斷門檻
prob = m.predict_proba(X[1500:])[:, 1]
th = 0.5  #@param {type:"number"}  # ← 只改這一行
pred = (prob >= th).astype(int)
tn, fp, fn, tp = confusion_matrix(y[1500:], pred).ravel()
prec = tp / (tp + fp) if tp + fp else 0
rec = tp / (tp + fn) if tp + fn else 0
print(th, '精確率', round(prec, 3), '召回率', round(rec, 3))

## 任務四：故意做一次資料洩漏，再看清楚它有多騙人

**怎麼做**：接著第 1 格的 `X`、`y` 跑。先看兩個分數差多少，再解釋為什麼。

`k1` 的訓練集偷偷含了 200 筆測試資料，`k2` 完全沒有。上面那個分數比較高，
**但它一點意義都沒有**——1-NN 會直接記住每一筆資料，混進去的那 200 筆，
最近的鄰居就是它自己。

分數忽然變得很漂亮，先懷疑洩漏。

In [ ]:
#@title 第 4 格：故意做一次資料洩漏
from sklearn.neighbors import KNeighborsClassifier
# k1 的訓練集偷偷含了 200 筆測試資料，k2 完全沒有
k1 = KNeighborsClassifier(1).fit(X[:1700], y[:1700])
k2 = KNeighborsClassifier(1).fit(X[:1500], y[:1500])
print('有洩漏 =', round(k1.score(X[1500:], y[1500:]), 3))
print('沒洩漏 =', round(k2.score(X[1500:], y[1500:]), 3))

## 延伸挑戰：A／B／C 三級

- **A（每組都要做完）換參數再跑**：把第 1 格的 `bad` 從 1 改成 20 再跑一次，
  記錄正確率與基準線的差距怎麼變。
- **B（每組都要做完）換做法再跑**：把第 4 格的 `X[:1700]` 改成 `X[:1600]`，
  看重疊筆數減半以後，虛高的分數少了多少。
- **C 說出為什麼**：說出訓練集混進測試資料為什麼會讓分數變高，並舉一個生活中的類比。

**手機上的常見狀況**：找不到 ▶ ＝沒切電腦版網站；改完欄位沒反應＝**要再按一次 ▶** 才生效；
第 2、3、4 格報 `NameError` ＝第 1 格沒跑成功，回去重跑第 1 格。

In [ ]:
#@title 收工檢查（直接按 ▶）
# ←投影片未含，執行所需
print('本週要交：四張截圖（比例、矩陣、門檻、洩漏）＋門檻紀錄表（四個門檻各一列）')
print('再寫三句：你的題目該看哪一個指標、為什麼、選哪一個門檻')
print('✅ 檔名：AI導論_W16_學號_姓名，本週表單週日 23:59 前繳交')